In [1]:
import pandas as pd
from pathlib import Path

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"

DISTRICT_OUTPUT = Path("data") / "exposure_district.csv"
BLOCK_OUTPUT = Path("data") / "exposure_block.csv"

df = pd.read_csv(INPUT_CSV)

# =============================================================================
# CLEAN KEYS
# =============================================================================

df["district"] = df["district"].astype(str).str.strip()
df["timeperiod"] = df["timeperiod"].astype(str).str.strip()

# =============================================================================
# Z-SCORE FUNCTION
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)
    if std == 0:
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std

# =============================================================================
# CLASSIFICATION
# =============================================================================

def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5

# =============================================================================
# DISTRICT MEAN POPULATION
# =============================================================================

district_df = (
    df.groupby(["district", "timeperiod"], as_index=False)
      .agg(
          district_population=("sum_population", "mean")
      )
)

# =============================================================================
# MONTHWISE Z-SCORE
# =============================================================================

district_df["population_z"] = (
    district_df.groupby("timeperiod")["district_population"]
    .transform(zscore)
)

# =============================================================================
# EXPOSURE CLASS
# =============================================================================

district_df["exposure"] = district_df["population_z"].apply(classify)

# =============================================================================
# SAVE DISTRICT OUTPUT
# =============================================================================

district_df.to_csv(DISTRICT_OUTPUT, index=False)

# =============================================================================
# APPEND DISTRICT EXPOSURE TO BLOCK DATA
# =============================================================================

block_df = df.copy()

if "exposure" in block_df.columns:
    block_df = block_df.drop(columns=["exposure"])

block_df = block_df.merge(
    district_df[
        [
            "district",
            "timeperiod",
            "district_population",
            "population_z",
            "exposure",
        ]
    ],
    on=["district", "timeperiod"],
    how="left",
    validate="many_to_one",
)

# =============================================================================
# SAVE BLOCK OUTPUT
# =============================================================================

block_df.to_csv(BLOCK_OUTPUT, index=False)

# =============================================================================
# SUMMARY
# =============================================================================

print(f"District file saved : {DISTRICT_OUTPUT}")
print(f"Block file saved    : {BLOCK_OUTPUT}")

print("\nDistrict rows:", len(district_df))
print("Block rows:", len(block_df))

print("\nExposure distribution:")
print(district_df["exposure"].value_counts().sort_index())

print("\nDistrict preview:")
print(
    district_df[
        [
            "district",
            "timeperiod",
            "district_population",
            "population_z",
            "exposure",
        ]
    ].head()
)

print("\nBlock preview:")
print(
    block_df[
        [
            "object_id",
            "block_name",
            "district",
            "timeperiod",
            "sum_population",
            "district_population",
            "exposure",
        ]
    ].head()
)

District file saved : data/exposure_district.csv
Block file saved    : data/exposure_block.csv

District rows: 1890
Block rows: 19782

Exposure distribution:
exposure
1     63
2    630
3    621
4    450
5    126
Name: count, dtype: int64

District preview:
  district timeperiod  district_population  population_z  exposure
0   Anugul    2021_01         175838.01123      0.524408         4
1   Anugul    2021_02         175838.01123      0.524408         4
2   Anugul    2021_03         175838.01123      0.524408         4
3   Anugul    2021_04         175838.01123      0.524408         4
4   Anugul    2021_05         175838.01123      0.524408         4

Block preview:
      object_id   block_name district timeperiod  sum_population  \
0  21-384-03276       ANUGUL   Anugul    2021_01   194589.039062   
1  21-384-03277    ATHMALLIK   Anugul    2021_01   141697.031250   
2  21-384-03278     BANARPAL   Anugul    2021_01   180526.906250   
3  21-384-03279  CHHENDIPADA   Anugul    2021_01   26